# Morning class 27/08 — Worksheet 08 SOLUTIONS: reading real files   (L05)

Every cell below was executed on the same Python the lab ships (3.13), against
the files in `data/`, and the quoted output is what it actually printed.

Q4, Q7 and Q9 each produce a number that is correct and easy to quote wrongly.
Those are the three to re-read.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — Reading real files. Run this once.
#
# `data/` sits next to this notebook, so these relative paths work whether you
# opened it in the shared console or in a git clone.

with open("data/orders.csv", encoding="utf-8") as fh:
    order_lines = fh.readlines()

with open("data/customers.csv", encoding="utf-8") as fh:
    customer_lines = fh.readlines()

with open("data/products.csv", encoding="utf-8") as fh:
    product_lines = fh.readlines()

with open("data/returns.csv", encoding="utf-8") as fh:
    return_lines = fh.readlines()

print("orders.csv   ", len(order_lines), "lines")
print("customers.csv", len(customer_lines), "lines")
print("products.csv ", len(product_lines), "lines")
print("returns.csv  ", len(return_lines), "lines")

PART A — What you actually got back

### Question 1

What `readlines()` gave you. -> the header as `'OrderID\tProductID\t…\tShippingCost\n'`, the first data row as `'8710\t657768\t40732966\t2009-01-04\t…\t1.93\n'`, then `1094 lines` and `1093 data rows`.

`readlines()` gives you a **list of strings**, one per line, and it keeps the
`\n` on the end of each. `print()` hides that; `repr()` shows it, which is
why you use `repr()` when you are working out what a file actually contains.

The `\t` between fields is a real tab character — these files are
tab-separated, not comma-separated, despite the `.csv` name. That is how
the course's data ships.

And `1094` is not the number of records. Line 0 is the header, so there are
1093. Nothing in the file marks that line as special — you have to know,
and every question below depends on you remembering.

In [ ]:
print(repr(order_lines[0]))
print(repr(order_lines[1]))

print(len(order_lines), "lines")
print(len(order_lines) - 1, "data rows")   # line 0 is the header

### Question 2

The columns, and the trailing newline. -> `0 OrderID` through `11 ShippingCost`, then `'1.93\n'` and `'1.93'`.

Printing the header with `enumerate` is the first thing to do with an
unfamiliar file — those indexes are what every later question uses, and
guessing them is how you end up totalling the discount column.

The last field of **every** row carries the line break, because that is what
separates it from the next row. `'1.93\n'` is not the same string as
`'1.93'`, and comparing it with `"1.93"` is `False`.

`.strip()` before `.split()` removes it. Do it once, at the point you split,
and you never think about it again. (`float('1.93\n')` happens to work —
float tolerates surrounding whitespace — which is exactly why this bites
you on a *text* column instead, months later.)

In [ ]:
header = order_lines[0].strip().split("\t")
for i, name in enumerate(header):
    print(i, name)

print("---")

raw = order_lines[1].split("\t")
print(repr(raw[-1]))                      # the \n is still on it

clean = order_lines[1].strip().split("\t")
print(repr(clean[-1]))

PART B — One pass, one answer

### Question 3

Customers who ordered. -> `307 customers ordered`, `1832 customers on file`, `1525 never ordered`.

A set is the whole answer to "how many distinct X" — add every value and
let it discard the repeats.

307 of 1832 ordered, so **83% of the customer file has no activity at all**.
That is normal for a dimension table and it is the reason `len(customers)`
is never the answer to "how many customers do we have", for any business
meaning of the question.

The `[1:]` on both loops is the header skip. Leave it off the customers
loop and you get 1833 and the string `CustomerID` counted as a person —
no error, one extra customer, and Q11 is where that finally goes bang.

In [ ]:
ordering = set()
for line in order_lines[1:]:
    ordering.add(line.strip().split("\t")[2])

on_file = set()
for line in customer_lines[1:]:
    on_file.add(line.strip().split("\t")[0])

print(len(ordering), "customers ordered")
print(len(on_file), "customers on file")
print(len(on_file) - len(ordering), "never ordered")

### Question 4

Lines against orders. -> `1093 order lines`, **`600 distinct orders`**, `329 orders have more than one line`, `biggest order: 6 lines`.

1093 rows, 600 orders. **`orders.csv` is one row per order *line*, not per
order** — if a customer bought four things, that is one order and four rows.

So `len(order_lines) - 1` is not the number of orders, and any average
computed with it is an average per line item. More than half the orders
here (329 of 600) have several lines, so the two numbers are not close.

This is `COUNT(*)` versus `COUNT(DISTINCT OrderID)`, in Python. If you did
24/08 you have already been caught by it once on the same table.

The counting loop is worksheet 02 Q11's, and the dictionary it builds gets
reused in Q9.

In [ ]:
lines_per_order = {}
for line in order_lines[1:]:
    oid = line.strip().split("\t")[0]
    if oid in lines_per_order:
        lines_per_order[oid] = lines_per_order[oid] + 1
    else:
        lines_per_order[oid] = 1

multi = 0
biggest = 0
for count in lines_per_order.values():
    if count > 1:
        multi = multi + 1
    if count > biggest:
        biggest = count

print(len(order_lines) - 1, "order lines")
print(len(lines_per_order), "distinct orders")
print(multi, "orders have more than one line")
print("biggest order:", biggest, "lines")

### Question 5

The highest unit price, without `max()`. -> `highest unit price: 3502.14`, `2 order lines at that price: ['394339', '394339']`.

Two passes, and both are necessary. The first finds *what the maximum is*;
the second finds *which rows have it*. You cannot do both at once, because
until the last row is read you do not know whether the current best will
survive.

Two lines came back — the **same ProductID twice**, on two different order
lines. A single-answer question (`max`) can have a multi-row answer, and
code that assumes one match silently reports whichever it happened to see
first.

`best = 0.0` as the starting point is fine here because prices are
positive. On data that can be negative it is a bug: start from the first
row's value instead, or `best = None` and special-case the first pass.

In [ ]:
best = 0.0
for line in order_lines[1:]:
    price = float(line.strip().split("\t")[10])
    if price > best:
        best = price

winners = []
for line in order_lines[1:]:
    parts = line.strip().split("\t")
    if float(parts[10]) == best:
        winners.append(parts[1])          # the ProductID

print("highest unit price:", best)
print(len(winners), "order lines at that price:", winners)

PART C — Joining two files

### Question 6

Joining to the product file. -> `394339 -> Okidata Pacemark 4410N Wide Format Dot Matrix Printer`; then **`1237 rows in products.csv`** but **`1234 keys in product_name`**; `558 products ordered`; `676 never ordered`.

**Three rows went in and never came out.** `products.csv` has 1237 data
rows and the dictionary has 1234 keys, because three `ProductID`s appear
twice and each second row overwrote the first.

They are not duplicates of each other, either — `481924` is both an
"Avery 501" and a "Boston 1799 Powerhouse Electric Pencil Sharpener";
`72479` is both "Accessory34" and "Laser & Ink Jet Business Envelopes".
Genuinely different products sharing an id. This is real data, not a
planted mistake, and `product_name[...]` will confidently give you the
wrong name for three of them.

A dictionary cannot hold a key twice, so building one from a file **assumes
the key is unique** — and it will not tell you when it isn't. That is
worksheet 03 Q7 again, on 1,237 rows instead of four. The one line that
catches it is the one above: compare the row count with the key count,
every time you build a lookup from a file.

(558 ordered of 1234, so 676 products have never been sold. Same shape as
Q3 — dimension tables carry rows that no fact ever references.)

In [ ]:
product_name = {}
for line in product_lines[1:]:
    parts = line.strip().split("\t")
    product_name[parts[0]] = parts[1]

for pid in sorted(set(winners)):
    print(pid, "->", product_name[pid])

ordered = set()
for line in order_lines[1:]:
    ordered.add(line.strip().split("\t")[1])

print(len(product_lines) - 1, "rows in products.csv")
print(len(product_name), "keys in product_name")
print(len(ordered), "products ordered")
print(len(product_name) - len(ordered), "never ordered")

### Question 7

By region, two ways. -> eight regions from `Atlantic` `144 lines / 85 orders` to `Yukon` `73 lines / 46 orders`; `Ontario` biggest at `302 lines / 164 orders`. **`TOTAL 1093 lines 640 orders`.**

Two things are wrong in that output and neither raises.

**`Prarie` is a typo in the source data** — it should be "Prairie". Group
by it and you get a region nobody will find by searching for the correct
spelling. Dirty dimension values are the normal case, and there is nothing
in the code that could have caught it; only reading the output could.

**The TOTAL says 640 orders and Q4 said 600.** The lines column adds up
perfectly — 1093, exactly the number of rows — but the orders column does
not, and it is 40 too high. The reason is that 39 `OrderID`s in this file
appear against more than one `CustomerID`, and 37 of those customers sit in
different regions, so those orders are counted once in each region they
touch.

**Distinct counts do not add up across groups.** Sums do, counts of rows
do, and counts of *distinct things* do not — because the same thing can
belong to two groups. That is true in SQL and in pandas exactly as it is
here, it is one of the most common wrong numbers in analytics, and the
only way to notice is to print the total and check it against a figure you
trust.

In [ ]:
customer_region = {}
for line in customer_lines[1:]:
    parts = line.strip().split("\t")
    customer_region[parts[0]] = parts[3]

lines_by_region = {}
orders_by_region = {}

for line in order_lines[1:]:
    parts = line.strip().split("\t")
    region = customer_region[parts[2]]
    if region in lines_by_region:
        lines_by_region[region] = lines_by_region[region] + 1
    else:
        lines_by_region[region] = 1
        orders_by_region[region] = set()
    orders_by_region[region].add(parts[0])

for region in sorted(lines_by_region):
    print("%-24s %5d lines %5d orders"
          % (region, lines_by_region[region], len(orders_by_region[region])))

print("%-24s %5d lines %5d orders"
      % ("TOTAL", sum(lines_by_region.values()),
         sum(len(v) for v in orders_by_region.values())))

PART D — Dates, without importing anything

### Question 8

Dates as text. -> `2009 290`, `2010 285`, `2011 252`, `2012 266`; then twelve months of 2012, from `2012-01 10` to `2012-12 23`, peaking at `2012-05 39`.

`"2012-05-14".split("-")` gives `['2012', '05', '14']`, and for grouping by
year or month that is the entire job — no parsing, no format string, no
import, and nothing that can hand you a timezone.

It works because the dates are in `YYYY-MM-DD`, which sorts correctly as
text: `sorted(by_month)` gives January to December precisely because the
months are zero-padded. `"9"` would have sorted after `"10"`, exactly as
in extra practice 01 Q6.

The moment you need weekday, date arithmetic or a different input format,
reach for `datetime` — that is what the course's own `Python Exercises 2`
does. Until then, do not.

Note these are line counts, not order counts. Q4's distinction applies to
every table on this sheet.

In [ ]:
by_year = {}
for line in order_lines[1:]:
    year = line.strip().split("\t")[3].split("-")[0]
    if year in by_year:
        by_year[year] = by_year[year] + 1
    else:
        by_year[year] = 1

for year in sorted(by_year):
    print(year, by_year[year])

print("---")

by_month = {}
for line in order_lines[1:]:
    date = line.strip().split("\t")[3]
    year, month, day = date.split("-")
    if year != "2012":
        continue
    if month in by_month:
        by_month[month] = by_month[month] + 1
    else:
        by_month[month] = 1

for month in sorted(by_month):
    print("2012-" + month, by_month[month])

PART E — The numbers, and their denominators

### Question 9

Returns, three denominators. -> `572 returned orders in returns.csv`, `67 of them appear in orders.csv`, `120 order lines belong to a returned order`; per line `0.10978956999085086`, per order `0.11166666666666666`, vs file `0.11713286713286714`.

Three percentages, around 11% each, and they are answers to three different
questions:

- **per line** (10.98%) — 120 of 1093 order *lines* were part of a returned
  order. Useful for warehouse handling, not for "how often do customers
  return things".
- **per order** (11.17%) — 67 of 600 orders came back. This is the return
  rate, and it is the one to report.
- **vs file** (11.71%) — 67 of the 572 returns in `returns.csv`. This is
  not a rate at all; it is *what fraction of all known returns happen to
  fall inside our sample of orders*, and it would change if you loaded a
  different slice of orders.

The third is the one worth being careful about, because it is the easiest
to compute by accident: `returns.csv` is right there, it has a length, and
dividing by it produces a plausible-looking 11.7%.

Note also that 505 of the 572 returned orders are **not in this orders
file** at all. That is expected — `returns.csv` covers the whole business
and `orders.csv` here is a 600-order sample — but if you had assumed every
return matched an order, you would have concluded the orders file was
missing 88% of its data.

In [ ]:
returned = set()
for line in return_lines[1:]:
    returned.add(line.strip().split("\t")[0])

returned_orders = set()
returned_lines = 0
for line in order_lines[1:]:
    oid = line.strip().split("\t")[0]
    if oid in returned:
        returned_orders.add(oid)
        returned_lines = returned_lines + 1

total_lines = len(order_lines) - 1
total_orders = len(lines_per_order)

print(len(returned), "returned orders in returns.csv")
print(len(returned_orders), "of them appear in orders.csv")
print(returned_lines, "order lines belong to a returned order")
print("per line: ", returned_lines / total_lines)
print("per order:", len(returned_orders) / total_orders)
print("vs file:  ", len(returned_orders) / len(returned))

### Question 10

Sales, and the reconciliation. -> `grand total: 1605576.217500002`; Ontario largest at `466003.99`, Nunavut smallest at `15890.71`; `sum of parts: 1605576.2174999998`; **`parts == whole: False`**.

Look at the last three lines. `1605576.217500002` against
`1605576.2174999998` — the same money, added in a different order, off by
about two millionths of a penny.

Nothing is wrong. Floating-point addition is **not associative**: adding
the rows one after another and adding them grouped by region round at
different points, and the tiny errors land differently. Worksheet 02 Q9
and worksheet 04 Q8 are the same fact at smaller scale.

So `parts == whole` is `False`, and a reconciliation check written as `==`
on floats will fail on correct data every time. Write it as
`abs(parts - whole) < 0.01`, or total in integer pence, or use
`decimal.Decimal` — the way you must never write it is the way that looks
most obvious.

The reconciliation itself is still worth doing. Off by 2e-6 is arithmetic;
off by 466003.99 is a region you dropped.

(Unlike Q7's order counts, the sales **do** add up — every row belongs to
exactly one region, and a sum of sums is a sum.)

In [ ]:
total_sales = 0.0
sales_by_region = {}

for line in order_lines[1:]:
    parts = line.strip().split("\t")
    amount = float(parts[6])
    total_sales = total_sales + amount
    region = customer_region[parts[2]]
    if region in sales_by_region:
        sales_by_region[region] = sales_by_region[region] + amount
    else:
        sales_by_region[region] = amount

print("grand total:", total_sales)
for region in sorted(sales_by_region):
    print("%-24s %14.2f" % (region, sales_by_region[region]))

parts_total = sum(sales_by_region.values())
print("sum of parts:", parts_total)
print("parts == whole:", parts_total == total_sales)

### Question 11

Forgetting the header. -> `ValueError: could not convert string to float: 'Sales'`.

`readlines()` does not know what a header is. Line 0 is a line like any
other, and `float("Sales")` is the first thing that notices.

This is the friendliest failure on the sheet, because it happens
immediately and says exactly what is wrong. Compare it with what would
happen if the column had been headed `0` rather than `Sales`: `float("0")`
is perfectly valid, the total would come out unchanged, and the **row
count** would be one too high — so every average computed from it would be
quietly, slightly wrong, forever.

That is the difference between a bug that costs you five minutes and one
that costs you a quarter of reporting. `[1:]` on every loop over a file
with a header, every time.

The general lesson of the sheet: nothing in a text file is typed, labelled
or validated. It is bytes with tabs in it. Every column is a string until
you convert it, the header is data until you skip it, keys are unique until
you check (Q6), and totals reconcile until you look (Q7, Q10).

In [ ]:
total = 0.0
for line in order_lines:          # no [1:] -- the header is in here
    total = total + float(line.strip().split("\t")[6])

# This is SUPPOSED to raise: ValueError: could not convert string to float:
# 'Sales'. The header row is just another line to readlines(); nothing marks
# it as special.
#
# If the column had been headed "0" instead of "Sales", float() would have
# accepted it, the total would have been off by nothing at all, and the ROW
# COUNT would have been off by one -- so every average computed from it would
# be quietly wrong. The error is the good outcome.
print(total)